# Ejercicio Módulo 5 - Dataset Swiss
**Inteligencia Artificial - CEIA - FIUBA**

**Daniel Alejandro Acero Varela**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

---

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [2]:
import pandas as pd

df = pd.read_csv("swiss (1).csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [3]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [4]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

##### COMPLETAR AQUI LO PEDIDO
print(X_train.shape, X_test.shape)
print(df.drop("Location", axis=1).corr()["Fertility"])
print(X.corr().round(2))

(23, 5) (24, 5)
Fertility           1.000000
Agriculture         0.353079
Examination        -0.645883
Education          -0.663789
Catholic            0.463685
Infant.Mortality    0.416556
Name: Fertility, dtype: float64
                  Agriculture  Examination  Education  Catholic  \
Agriculture              1.00        -0.69      -0.64      0.40   
Examination             -0.69         1.00       0.70     -0.57   
Education               -0.64         0.70       1.00     -0.15   
Catholic                 0.40        -0.57      -0.15      1.00   
Infant.Mortality        -0.06        -0.11      -0.10      0.18   

                  Infant.Mortality  
Agriculture                  -0.06  
Examination                  -0.11  
Education                    -0.10  
Catholic                      0.18  
Infant.Mortality              1.00  


In [7]:
def evaluar(y_true, y_pred) -> dict:
    """Devuelve un dict con MAE, MSE y R2."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

In [8]:
print(evaluar(y, y))
print(evaluar(y, np.full(len(y), y.mean())))

{'MAE': 0.0, 'MSE': 0.0, 'R2': 1.0}
{'MAE': 9.577818017202357, 'MSE': 152.72244454504298, 'R2': 0.0}


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema.
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [9]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
>
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [10]:
alphas = np.logspace(-4, 1, 500)

##### COMPLETAR AQUI LO PEDIDO
pipe_lineal = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

#print(pipe_lineal)
y_pred_lineal = pipe_lineal.fit(X_train, y_train).predict(X_test)
print("Regresión Lineal:", evaluar(y_test, y_pred_lineal))
print(y_pred_lineal.shape)

#print(evaluar(y_test, y_pred))
pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=alphas, cv=3))
])

pipe_lasso.fit(X_train, y_train)
y_pred_lasso = pipe_lasso.predict(X_test)

print("Mejor alpha encontrado por LassoCV :", pipe_lasso.named_steps["model"].alpha_)
print("Lasso :", evaluar(y_test, y_pred_lasso))

print(pd.Series(pipe_lasso.named_steps["model"].coef_, index=X.columns).sort_values())


Regresión Lineal: {'MAE': 6.093707279834089, 'MSE': 64.45719426896419, 'R2': 0.5744462918768746}
(24,)
Mejor alpha encontrado por LassoCV : 2.18110892419152
Lasso : {'MAE': 5.99540457594174, 'MSE': 64.29431837743248, 'R2': 0.5755216169882339}
Education          -2.404905
Examination        -1.762433
Agriculture        -0.000000
Catholic            1.719649
Infant.Mortality    3.200407
dtype: float64


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        | MAE | MSE | $R^2$ |
| ----------------------------- | --- | --- | ----- |
| Regresión Lineal              | 6.094 |  64.457   |   0.5744    |
| Modelo Regularizado L1 |  5.995   |  64.294   |   0.5755    |


> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

El modelo Lasso parece ser ligeramente mejor que el de regresión lineal aunque sólo tenemos 24 observaciones de test lo cual no le ayuda a ser consistente. El resultado es un rendimiento predictivo muy similar. Por otra parte, Lasso eliminó  `Agriculture` obteniendo un mejor desempeño con diferencias mínimas, un modelo más simple, de 4 variables, además de contar con selección automática de variables y menor riesgo de sobreajuste.